# Week 3: Generation Baseline Pipeline

In [20]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

## 2. BM25 Retrieval Test

In [21]:
from bm25_retriever import BM25Retriever

In [22]:
corpus = [
    "Canberra is the capital city of Australia.",
    "Sydney is the largest city in Australia.",
    "Australia is a country in Oceania."
]

retriever = BM25Retriever(corpus)

query = "what is the capital of australia"

results = retriever.retrieve(query, k=3)

results

[{'score': 1.0204140028176778,
  'passage': 'Canberra is the capital city of Australia.'},
 {'score': 0.5462687528173171,
  'passage': 'Australia is a country in Oceania.'},
 {'score': 0.021244078581021307,
  'passage': 'Sydney is the largest city in Australia.'}]

## 3. Load Generation Model

In [23]:
!pip install transformers torch sentencepiece evaluate rouge_score sacrebleu

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [25]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Using device:", device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Using device: cpu


## 4. Build Retrieval-Augmented Generation Pipeline

In [26]:
def build_context(results):
    passages = [r["passage"] for r in results]
    return " ".join(passages)

In [27]:
def generate_answer(query, results):

    context = build_context(results)

    input_text = f"question: {query} context: {context}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

## 5. Demo Example

In [28]:
query = "what is the capital of australia"

results = retriever.retrieve(query, k=3)

answer = generate_answer(query, results)

print("Query:", query)
print("Retrieved passages:")

for r in results:
    print("-", r["passage"])

print("\nGenerated answer:", answer)

Query: what is the capital of australia
Retrieved passages:
- Canberra is the capital city of Australia.
- Australia is a country in Oceania.
- Sydney is the largest city in Australia.

Generated answer: Canberra is the capital city of Australia. Australia is a country in Oceania


## Run generation on evaluation queries

In [29]:
eval_queries = [
    "what is the capital of australia",
    "where is the eiffel tower located",
    "who invented the telephone"
]

references = [
    "Canberra",
    "Paris",
    "Alexander Graham Bell"
]

predictions = []

for query in eval_queries:

    results = retriever.retrieve(query, k=3)

    answer = generate_answer(query, results)

    predictions.append(answer)

## 7. Automatic Evaluation

In [30]:
import evaluate

In [31]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

In [32]:
rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

bleu_result = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("ROUGE-L:", rouge_result["rougeL"])
print("BLEU:", bleu_result["bleu"])

ROUGE-L: 0.04761904761904762
BLEU: 0.0


## 8. Example Outputs

In [33]:
for i, query in enumerate(eval_queries):

    print("Query:", query)
    print("Reference:", references[i])
    print("Prediction:", predictions[i])
    print()

Query: what is the capital of australia
Reference: Canberra
Prediction: Canberra is the capital city of Australia. Australia is a country in Oceania

Query: where is the eiffel tower located
Reference: Paris
Prediction: Sydney is the largest city in Australia

Query: who invented the telephone
Reference: Alexander Graham Bell
Prediction: the telephone



## 9. Discussion

The baseline generation pipeline demonstrates that combining BM25 retrieval with a pretrained T5 model can produce plausible answers.

However, the automatic evaluation scores remain low due to several factors:

1. The generation model is not fine-tuned on MS MARCO QA data.
2. The retrieval corpus used in this demonstration is extremely small.
3. Reference answers are often very short, which negatively affects BLEU scores.

Future work will focus on improving retrieval quality and applying model fine-tuning.